In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LassoCV, RidgeCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

FILE = "Water Quality.csv"
OUT = "task3_feature_selection_outputs"
TARGET = None  # Set only to a valid, scientifically justified numeric outcome, e.g. a calculated WQI.
MAX_BASE_FEATURES = 12

def main():
    if not Path(FILE).exists(): raise FileNotFoundError(f"Cannot find {FILE!r}; set FILE to full path.")
    Path(OUT).mkdir(exist_ok=True)
    df = pd.read_csv(FILE, low_memory=False)
    req = {"ActivityStartDate","CharacteristicName","ResultValue"}
    if req-set(df.columns): raise ValueError(f"Missing columns: {req-set(df.columns)}")
    df["ActivityStartDate"] = pd.to_datetime(df.ActivityStartDate, errors="coerce")
    df["ResultValue"] = pd.to_numeric(df.ResultValue, errors="coerce")
    df = df.dropna(subset=["ActivityStartDate","CharacteristicName","ResultValue"])
    df["Month"] = df.ActivityStartDate.dt.to_period("M").dt.to_timestamp()
    keys = [c for c in ["MonitoringLocationID","Month"] if c in df]
    wide = df.pivot_table(index=keys, columns="CharacteristicName", values="ResultValue", aggfunc="median").reset_index()
    wide.columns.name = None
    wide.to_csv(Path(OUT)/"monthly_wide_features.csv", index=False)
    num = wide.select_dtypes(include=np.number)
    candidates = [c for c in num if num[c].notna().sum() >= 10 and num[c].nunique(dropna=True)>1]
    pd.DataFrame({"numeric_candidate_columns":candidates}).to_csv(Path(OUT)/"numeric_candidates.csv", index=False)
    if TARGET is None:
        print("Created monthly_wide_features.csv and numeric_candidates.csv.")
        print("TARGET is None: no model fit because the dataset does not inherently contain a validated WQI outcome.")
        print("Set TARGET only after you define or provide a scientifically justified numeric target.")
        return
    if TARGET not in num: raise ValueError(f"TARGET {TARGET!r} is not a numeric column in the wide table.")
    features = [c for c in candidates if c != TARGET][:MAX_BASE_FEATURES]
    if len(features)<2: raise ValueError("At least two predictors are needed.")
    model = wide[features+[TARGET]].replace([np.inf,-np.inf],np.nan).dropna(subset=[TARGET])
    if len(model)<20: raise ValueError("Need at least 20 target observations for this example.")
    X, y = model[features], model[TARGET]
    imp = SimpleImputer(strategy="median")
    Xi = imp.fit_transform(X)
    poly = PolynomialFeatures(degree=2, include_bias=False)
    Xp = poly.fit_transform(Xi)
    names = poly.get_feature_names_out(features)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(Xp)
    lasso = LassoCV(cv=5, max_iter=20000, random_state=42).fit(Xs,y)
    ridge = RidgeCV(alphas=np.logspace(-3,3,25)).fit(Xs,y)
    coef = pd.DataFrame({"feature":names,"lasso_coefficient":lasso.coef_,
                         "absolute_lasso_coefficient":np.abs(lasso.coef_),"ridge_coefficient":ridge.coef_})
    coef.sort_values("absolute_lasso_coefficient",ascending=False).to_csv(Path(OUT)/"regularized_feature_coefficients.csv",index=False)
    cv = KFold(n_splits=5,shuffle=True,random_state=42)
    pipe = Pipeline([("imputer",SimpleImputer(strategy="median")),
                     ("poly",PolynomialFeatures(degree=2,include_bias=False)),
                     ("scale",StandardScaler()),("ridge",RidgeCV(alphas=np.logspace(-3,3,25)))])
    scores = cross_val_score(pipe,X,y,cv=cv,scoring="r2")
    pd.DataFrame([{"target":TARGET,"rows":len(y),"base_features":len(features),
                   "polynomial_features":len(names),"lasso_alpha":lasso.alpha_,"ridge_alpha":ridge.alpha_,
                   "cv_r2_mean":scores.mean(),"cv_r2_std":scores.std()}]).to_csv(Path(OUT)/"model_summary.csv",index=False)
    print("Saved regularized coefficients and model summary. Use time-aware validation for temporal prediction.")
if __name__ == "__main__": main()


Created monthly_wide_features.csv and numeric_candidates.csv.
TARGET is None: no model fit because the dataset does not inherently contain a validated WQI outcome.
Set TARGET only after you define or provide a scientifically justified numeric target.
